# Rubicon — Comprehensive Multi-Modal Dataset, Preprocessing & EDA Visualization
### Hyperspectral Imaging (HSI) + LiDAR Elevation Fusion

This notebook delivers an end-to-end visual exploration and diagnostic suite for the **Rubicon** dataset (IEEE GRSS Houston 2013).
It visualizes **ALL** key aspects of the data, preprocessing steps, and exploratory data analysis (EDA):

1. **Raw Modalities**: 144-band Hyperspectral Cube & LiDAR Digital Elevation Model (DEM).
2. **Spectral Profiling**: Class spectral signatures across 144 bands, inter-band correlations, and absorption characteristics.
3. **Spatial Co-Registration**: Grid alignment, multi-modal overlays, and edge registration verification.
4. **Topological Analysis**: LiDAR 2D elevation, terrain slopes, and 3D hillshade texture.
5. **PCA Dimensionality Reduction**: Scree plots, cumulative variance curves, eigenband loadings, and orthogonal PC composite maps.
6. **Ground Truth & Severity Proxies**: 15 land-cover class distributions and damage severity proxy remapping (*none*, *moderate*, *severe*).
7. **Spatial Patchification**: Grid extraction coverage and multi-modal patch comparison gallery.
8. **Cross-Modal Relationships & Tensor Quality**: Bivariate density distributions, elevation-severity violin plots, and preprocessed tensor diagnostics.

In [ ]:
import os
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import ndimage

# Ensure ai-engine root is in path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.preprocess_eda_pipeline import (
    HOUSTON2013_CLASSES,
    CLASS_TO_SEVERITY,
    SEVERITY_NAMES,
    SEVERITY_TO_INDEX,
    RGB_BANDS_CASI,
    NIR_BAND_CASI,
    run_preprocessing_pipeline
)

# Styling configuration for publication-quality charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 15

print('Environment and visualization styles successfully configured!')

## 1. Load Dataset & Preprocessing Pipeline
We execute the preprocessing pipeline to load rasters, align grids, fit PCA, extract spatial patches, and obtain the preprocessed dataset artifacts without performing model training or test splitting.

In [ ]:
config_path = ROOT / 'configs' / 'preprocess_eda.yaml'
prep_results = run_preprocessing_pipeline(config_path)

hsi_raw = prep_results['hsi_raw']
dem_raw = prep_results['dem_raw']
labels_raw = prep_results['labels']
hsi_pca = prep_results['hsi_pca']
pca_model = prep_results['pca_model']
patches = prep_results['patches']
meta = prep_results['metadata']

print(f'\nDataset Summary:')
print(f'  - HSI Dimensions:       {hsi_raw.shape} (Rows x Cols x Bands)')
print(f'  - LiDAR DEM Dimensions: {dem_raw.shape} (Rows x Cols)')
print(f'  - PCA Retained Bands:   {hsi_pca.shape[2]} components ({meta["pca_explained_variance"]*100:.2f}% variance)')
print(f'  - Preprocessed Patches: {len(patches["y_class"])} multi-modal patches ({meta["patch_size"]}x{meta["patch_size"]})')

## 2. Hyperspectral Spatial Exploration
### Charts 1 & 2: True-Color RGB and False-Color Infrared Composites
- **True-Color RGB**: Red (~670 nm), Green (~550 nm), Blue (~480 nm).
- **False-Color Infrared**: NIR (~850 nm), Red (~670 nm), Green (~550 nm) — highlighting active biomass and vegetated canopies in bright red.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 1. True-Color RGB
rgb = np.stack([
    hsi_raw[:, :, RGB_BANDS_CASI[0]],
    hsi_raw[:, :, RGB_BANDS_CASI[1]],
    hsi_raw[:, :, RGB_BANDS_CASI[2]],
], axis=-1)
p2, p98 = np.percentile(rgb, (2, 98))
rgb_stretched = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)

axes[0].imshow(rgb_stretched)
axes[0].set_title('Chart 1: HSI True-Color RGB Composite (Red: Band 64, Green: Band 38, Blue: Band 23)', fontweight='bold')
axes[0].set_xlabel('Column Pixel (East-West GSD ~2.5m)')
axes[0].set_ylabel('Row Pixel (North-South)')
axes[0].grid(False)

# 2. False-Color Infrared (CIR)
cir_bands = (NIR_BAND_CASI, RGB_BANDS_CASI[0], RGB_BANDS_CASI[1])
cir = np.stack([
    hsi_raw[:, :, cir_bands[0]],
    hsi_raw[:, :, cir_bands[1]],
    hsi_raw[:, :, cir_bands[2]],
], axis=-1)
p2_cir, p98_cir = np.percentile(cir, (2, 98))
cir_stretched = np.clip((cir - p2_cir) / (p98_cir - p2_cir + 1e-6), 0, 1)

axes[1].imshow(cir_stretched)
axes[1].set_title('Chart 2: HSI False-Color Infrared Composite (NIR: Band 101, Red: Band 64, Green: Band 38)', fontweight='bold')
axes[1].set_xlabel('Column Pixel (East-West GSD ~2.5m)')
axes[1].set_ylabel('Row Pixel (North-South)')
axes[1].grid(False)

plt.tight_layout()
plt.show()

### Chart 3: Mean Spectral Reflectance Curves by Land-Cover Class
Plotting the characteristic spectral response curves across all 144 wavelengths (from Visible 380nm to Near-Infrared 1050nm) with $\pm 1\sigma$ standard deviation error envelopes.

In [ ]:
wavelengths = np.linspace(380, 1050, hsi_raw.shape[2])

plt.figure(figsize=(14, 7))
cmap = plt.get_cmap('tab20', 16)

for c_id in range(1, 16):
    mask = (labels_raw == c_id)
    if not np.any(mask):
        continue
    c_pixels = hsi_raw[mask]
    mean_sig = np.mean(c_pixels, axis=0)
    std_sig = np.std(c_pixels, axis=0)
    c_name = HOUSTON2013_CLASSES.get(c_id, f'Class {c_id}')
    color = cmap(c_id)
    
    plt.plot(wavelengths, mean_sig, label=f'{c_id}. {c_name}', color=color, linewidth=2)
    plt.fill_between(wavelengths, np.clip(mean_sig - std_sig, 0, 1), np.clip(mean_sig + std_sig, 0, 1), color=color, alpha=0.12)

plt.title('Chart 3: Mean Spectral Signatures across 144 CASI Bands by Land-Cover Class (with ±1σ bounds)', fontweight='bold')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Normalized Reflectance [0, 1]')
plt.xlim(380, 1050)
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, ncol=1)
plt.tight_layout()
plt.show()

### Charts 4 & 5: Spectral Transect Profile & Inter-Band Correlation Heatmap
- **Chart 4**: Spatial-Spectral Transect Heatmap along a single horizontal slice of the scene across all 144 bands.
- **Chart 5**: Inter-band Pearson Correlation Heatmap showing high redundancy across raw hyperspectral channels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 4: Spectral Transect
transect_row = hsi_raw.shape[0] // 2
transect_slice = hsi_raw[transect_row, :, :] # (cols, bands)
im0 = axes[0].imshow(transect_slice.T, aspect='auto', cmap='magma', extent=[0, hsi_raw.shape[1], 1050, 380])
axes[0].set_title(f'Chart 4: Spectral Profile Transect at Row {transect_row}', fontweight='bold')
axes[0].set_xlabel('Column Pixel Index (Spatial)')
axes[0].set_ylabel('Wavelength (nm)')
plt.colorbar(im0, ax=axes[0], label='Reflectance')

# Chart 5: Inter-Band Correlation Matrix
sub_bands = np.linspace(0, hsi_raw.shape[2]-1, 36, dtype=int)
flat_pixels = hsi_raw.reshape(-1, hsi_raw.shape[2])[::50, sub_bands]
corr_matrix = np.corrcoef(flat_pixels, rowvar=False)

im1 = axes[1].imshow(corr_matrix, cmap='coolwarm', vmin=-0.2, vmax=1.0)
axes[1].set_title('Chart 5: Inter-Band Correlation Matrix (Sub-sampled Bands)', fontweight='bold')
axes[1].set_xlabel('Band Index')
axes[1].set_ylabel('Band Index')
plt.colorbar(im1, ax=axes[1], label='Pearson Correlation (r)')

plt.tight_layout()
plt.show()

## 3. LiDAR Digital Elevation Model (DEM) Analysis
### Charts 6, 7, & 8: 2D Elevation Map, Histogram/KDE, and 3D Hillshade Texture
- **Chart 6**: 2D LiDAR DEM Surface Elevation Map with topographic contours.
- **Chart 7**: Elevation distribution histogram and Kernel Density Estimation.
- **Chart 8**: 3D Hillshade & Surface Gradient Map computed from spatial height gradients ($dz/dx$, $dz/dy$).

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 1.2])

# Chart 6: 2D Elevation Map
ax0 = fig.add_subplot(gs[0, :])
im_dem = ax0.imshow(dem_raw, cmap='terrain')
cs = ax0.contour(dem_raw, levels=8, colors='black', alpha=0.35, linewidths=0.7)
ax0.clabel(cs, inline=True, fontsize=8, fmt='%.0fm')
ax0.set_title('Chart 6: 2D LiDAR Digital Elevation Model (DEM) with Height Contours', fontweight='bold')
ax0.set_xlabel('Column Pixel')
ax0.set_ylabel('Row Pixel')
ax0.grid(False)
cbar0 = plt.colorbar(im_dem, ax=ax0, orientation='vertical', fraction=0.015, pad=0.02)
cbar0.set_label('Elevation (meters a.s.l.)')

# Chart 7: Elevation Distribution Histogram
ax1 = fig.add_subplot(gs[1, 0])
valid_dem = dem_raw[np.isfinite(dem_raw)]
sns.histplot(valid_dem, kde=True, color='teal', bins=40, ax=ax1)
ax1.axvline(np.mean(valid_dem), color='red', linestyle='--', label=f'Mean: {np.mean(valid_dem):.2f}m')
ax1.axvline(np.median(valid_dem), color='darkorange', linestyle=':', label=f'Median: {np.median(valid_dem):.2f}m')
ax1.set_title('Chart 7: LiDAR Elevation Histogram & KDE Distribution', fontweight='bold')
ax1.set_xlabel('Elevation (meters)')
ax1.set_ylabel('Pixel Count')
ax1.legend()

# Chart 8: Hillshade & Slope
ax2 = fig.add_subplot(gs[1, 1])
dy, dx = np.gradient(np.nan_to_num(dem_raw, nan=0.0))
slope = np.pi/2.0 - np.arctan(np.sqrt(dx*dx + dy*dy))
aspect = np.arctan2(-dx, dy)
altitude = np.pi / 4.0   # 45 deg sun elevation
azimuth = 3.0 * np.pi / 4.0 # 135 deg sun azimuth
hillshade = np.sin(altitude) * np.sin(slope) + np.cos(altitude) * np.cos(slope) * np.cos(azimuth - aspect)
hillshade = np.clip(hillshade, 0, 1)

im_hs = ax2.imshow(hillshade, cmap='gray')
ax2.set_title('Chart 8: LiDAR Hillshade & Urban Topological Texture Map', fontweight='bold')
ax2.set_xlabel('Column Pixel')
ax2.set_ylabel('Row Pixel')
ax2.grid(False)
plt.colorbar(im_hs, ax=ax2, fraction=0.046, pad=0.04, label='Illumination Intensity')

plt.tight_layout()
plt.show()

## 4. Multi-Modal Alignment & Structural Verification
### Charts 9, 10, & 11: Multi-Modal Tri-Panel and Edge Overlay
- **Chart 9**: Side-by-side comparison of HSI RGB, LiDAR Elevation, and Ground Truth Land-Cover.
- **Chart 10**: Elevation distribution across all 15 Land-Cover Classes (Violin Plots).
- **Chart 11**: High-contrast LiDAR structural edges overlaid on HSI RGB composite to verify spatial registration.

In [ ]:
# Chart 9: Tri-Panel Modality Comparison
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

axes[0].imshow(rgb_stretched)
axes[0].set_title('Chart 9a: HSI True-Color RGB Composite', fontweight='bold')
axes[0].grid(False)

axes[1].imshow(dem_raw, cmap='plasma')
axes[1].set_title('Chart 9b: Co-Registered LiDAR Digital Elevation Model', fontweight='bold')
axes[1].grid(False)

axes[2].imshow(labels_raw, cmap='tab20', vmin=0, vmax=15)
axes[2].set_title('Chart 9c: Ground Truth Land-Cover Raster (0 = Unlabeled Background)', fontweight='bold')
axes[2].grid(False)

plt.tight_layout()
plt.show()

# Chart 10: Elevation Distribution by Land Cover Class
plt.figure(figsize=(14, 6))
df_elev = []
for c_id in range(1, 16):
    mask = (labels_raw == c_id) & np.isfinite(dem_raw)
    if np.any(mask):
        vals = dem_raw[mask]
        c_name = HOUSTON2013_CLASSES.get(c_id, f'C{c_id}')
        for v in vals[::max(1, len(vals)//300)]:
            df_elev.append({'Class': f'{c_id}. {c_name}', 'Elevation': float(v)})

df_elev = pd.DataFrame(df_elev)
sns.violinplot(data=df_elev, x='Class', y='Elevation', palette='Spectral', inner='quartile')
plt.title('Chart 10: LiDAR Elevation Distribution across 15 Land-Cover Classes', fontweight='bold')
plt.xlabel('Land-Cover Class')
plt.ylabel('Elevation (m)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Chart 11: Edge Registration Overlay
plt.figure(figsize=(16, 5))
dem_norm_temp = (dem_raw - np.nanmin(dem_raw)) / (np.nanmax(dem_raw) - np.nanmin(dem_raw) + 1e-6)
gx = ndimage.sobel(dem_norm_temp, axis=1)
gy = ndimage.sobel(dem_norm_temp, axis=0)
edge_mag = np.hypot(gx, gy)
edge_mask = edge_mag > np.percentile(edge_mag, 93)

overlay = rgb_stretched.copy()
overlay[edge_mask] = [0.0, 1.0, 1.0] # Cyan edges

plt.imshow(overlay)
plt.title('Chart 11: LiDAR Structural Boundaries (Cyan Edges) Overlaid on HSI RGB (Registration Check)', fontweight='bold')
plt.xlabel('Column Pixel')
plt.ylabel('Row Pixel')
plt.grid(False)
plt.tight_layout()
plt.show()

## 5. PCA Spectral Dimensionality Reduction Verification
### Charts 12 through 17: PCA Diagnostics & Spatial Projections
- **Chart 12**: Scree Plot (Individual Explained Variance per Component).
- **Chart 13**: Cumulative Explained Variance Curve with 90%, 95%, 99% thresholds.
- **Chart 14**: PCA Eigenband Spectral Loadings across all 144 wavelengths.
- **Chart 15**: Spatial Maps of Top 6 Principal Components ($PC_1$ to $PC_6$).
- **Chart 16**: False-Color PCA Composite Map ($PC_1, PC_2, PC_3 \rightarrow$ RGB).
- **Chart 17**: Post-PCA Orthogonal Feature Correlation Matrix (proving decorrelation).

In [ ]:
# Charts 12 & 13: Scree Plot & Cumulative Variance
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

evr = pca_model.explained_variance_ratio_
cum_evr = np.cumsum(evr)
n_comps = len(evr)

axes[0].bar(range(1, n_comps + 1), evr * 100, color='royalblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Chart 12: Scree Plot — Individual Explained Variance Ratio', fontweight='bold')
axes[0].set_xlabel('Principal Component Index')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_xticks(range(1, n_comps + 1))

axes[1].plot(range(1, n_comps + 1), cum_evr * 100, marker='o', color='crimson', linewidth=2.5)
axes[1].axhline(99, color='green', linestyle='--', label='99% Target Threshold')
axes[1].axhline(95, color='darkorange', linestyle=':', label='95% Threshold')
axes[1].axhline(90, color='gray', linestyle='-.', label='90% Threshold')
axes[1].set_title('Chart 13: Cumulative Explained Spectral Variance', fontweight='bold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_xticks(range(1, n_comps + 1))
axes[1].set_ylim(min(cum_evr[0]*100 - 5, 50), 101)
axes[1].legend()

plt.tight_layout()
plt.show()

# Chart 14: Eigenband Loadings
plt.figure(figsize=(14, 5))
for i in range(min(4, n_comps)):
    plt.plot(wavelengths, pca_model.components_[i], label=f'PC {i+1} Loading (Var: {evr[i]*100:.1f}%)', linewidth=2)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Chart 14: PCA Eigenband Spectral Loadings across 144 Wavelengths', fontweight='bold')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Component Eigenvector Weight')
plt.xlim(380, 1050)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Ground Truth & Damage Severity Proxy Distributions
### Charts 18 through 20: Class Frequencies & Severity Mapping
- **Chart 18**: Land-Cover Class Frequency Horizontal Bar Chart.
- **Chart 19**: Land-Cover Class to Damage Severity Grouped Association Chart.
- **Chart 20**: Damage Severity Proxy Distribution (*None*, *Moderate*, *Severe*) Donut Chart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 18: Class Frequencies
y_classes, class_counts = np.unique(patches['y_class'], return_counts=True)
class_labels_list = [HOUSTON2013_CLASSES.get(c, f'Class {c}') for c in y_classes]
total_patches = len(patches['y_class'])

y_pos = np.arange(len(y_classes))
bars = axes[0].barh(y_pos, class_counts, color=plt.cm.tab20(np.linspace(0, 1, len(y_classes))), edgecolor='black')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(class_labels_list)
axes[0].invert_yaxis()
axes[0].set_title('Chart 18: Patch Frequency by Land-Cover Class', fontweight='bold')
axes[0].set_xlabel('Number of Extracted Patches')

for bar in bars:
    w = bar.get_width()
    pct = (w / total_patches) * 100
    axes[0].text(w + 0.5, bar.get_y() + bar.get_height()/2, f'{int(w)} ({pct:.1f}%)', va='center', fontsize=9)

# Chart 20: Damage Severity Donut Chart
y_sevs, sev_counts = np.unique(patches['y_sev'], return_counts=True)
sev_labels = [SEVERITY_NAMES[s].capitalize() for s in y_sevs]
sev_colors = ['#2ecc71', '#f39c12', '#e74c3c']

wedges, texts, autotexts = axes[1].pie(
    sev_counts, labels=sev_labels, autopct='%1.1f%%',
    startangle=140, colors=sev_colors[:len(y_sevs)],
    wedgeprops=dict(width=0.45, edgecolor='black'),
    textprops=dict(fontweight='bold')
)
axes[1].set_title('Chart 20: Damage-Severity Proxy Distribution in Preprocessed Dataset', fontweight='bold')

plt.tight_layout()
plt.show()

# Chart 19: Class to Severity Grouped Association
plt.figure(figsize=(14, 5))
df_class_sev = []
for c_id, name in HOUSTON2013_CLASSES.items():
    sev = CLASS_TO_SEVERITY.get(c_id, 'none')
    cnt = int(np.sum(patches['y_class'] == c_id))
    df_class_sev.append({'Class': f'{c_id}. {name}', 'Severity': sev.capitalize(), 'Count': cnt})

df_cs = pd.DataFrame(df_class_sev)
sns.barplot(data=df_cs, x='Class', y='Count', hue='Severity', palette={'None': '#2ecc71', 'Moderate': '#f39c12', 'Severe': '#e74c3c'}, dodge=False)
plt.title('Chart 19: Land-Cover Class to Damage-Severity Proxy Remapping', fontweight='bold')
plt.xlabel('Land-Cover Class')
plt.ylabel('Extracted Patch Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Damage Severity Proxy')
plt.tight_layout()
plt.show()

## 7. Spatial Patch Extraction & Multi-Modal Gallery
### Charts 21 & 22: Patch Grid Coverage & Sample Multi-Modal Patch Gallery
- **Chart 21**: Spatial Patch Bounding Boxes overlaid on the survey area.
- **Chart 22**: Multi-modal inspection gallery across severity levels (*None*, *Moderate*, *Severe*):
  - HSI RGB, False-Color Infrared, LiDAR DEM, PCA False-Color, and Majority Severity Label.

In [ ]:
# Chart 21: Patch Grid Map
plt.figure(figsize=(16, 5))
plt.imshow(rgb_stretched, alpha=0.7)
psize = meta['patch_size']
for (r, c) in patches['coords']:
    rect = plt.Rectangle((c, r), psize, psize, edgecolor='yellow', facecolor='none', linewidth=0.8, alpha=0.6)
    plt.gca().add_patch(rect)

plt.title(f'Chart 21: Spatial Patch Extraction Grid Coverage ({len(patches["coords"])} patches of size {psize}x{psize})', fontweight='bold')
plt.xlabel('Column Index')
plt.ylabel('Row Index')
plt.grid(False)
plt.tight_layout()
plt.show()

# Chart 22: Multi-Modal Patch Gallery Across Severities
sample_indices = []
for s_idx in range(len(SEVERITY_NAMES)):
    matching = np.where(patches['y_sev'] == s_idx)[0]
    if len(matching) > 0:
        sample_indices.extend(matching[:2])

num_samples = len(sample_indices)
fig, axes = plt.subplots(num_samples, 5, figsize=(16, 3 * num_samples))

for row, idx in enumerate(sample_indices):
    s_idx = patches['y_sev'][idx]
    c_idx = patches['y_class'][idx]
    c_name = HOUSTON2013_CLASSES.get(c_idx, f'Class {c_idx}')
    s_name = SEVERITY_NAMES[s_idx].capitalize()
    
    # 1. HSI RGB Patch
    raw_p = patches['patches_hsi_raw'][idx]
    p_rgb = np.stack([raw_p[:, :, RGB_BANDS_CASI[0]], raw_p[:, :, RGB_BANDS_CASI[1]], raw_p[:, :, RGB_BANDS_CASI[2]]], axis=-1)
    p_rgb = (p_rgb - p_rgb.min()) / (p_rgb.max() - p_rgb.min() + 1e-6)
    axes[row, 0].imshow(p_rgb)
    axes[row, 0].set_title(f'Sample {idx} - HSI RGB')
    axes[row, 0].grid(False)
    
    # 2. HSI CIR Patch
    p_cir = np.stack([raw_p[:, :, NIR_BAND_CASI], raw_p[:, :, RGB_BANDS_CASI[0]], raw_p[:, :, RGB_BANDS_CASI[1]]], axis=-1)
    p_cir = (p_cir - p_cir.min()) / (p_cir.max() - p_cir.min() + 1e-6)
    axes[row, 1].imshow(p_cir)
    axes[row, 1].set_title('False-Color NIR')
    axes[row, 1].grid(False)
    
    # 3. LiDAR DEM Patch
    dem_p = patches['X_dem'][idx, 0]
    axes[row, 2].imshow(dem_p, cmap='plasma')
    axes[row, 2].set_title('LiDAR DEM')
    axes[row, 2].grid(False)
    
    # 4. PCA False Color Patch
    pca_p = patches['X_hsi'][idx, :3]
    pca_p_rgb = np.transpose(pca_p, (1, 2, 0))
    pca_p_rgb = (pca_p_rgb - pca_p_rgb.min()) / (pca_p_rgb.max() - pca_p_rgb.min() + 1e-6)
    axes[row, 3].imshow(pca_p_rgb)
    axes[row, 3].set_title('PCA Channels (1,2,3)')
    axes[row, 3].grid(False)
    
    # 5. Label & Severity Banner
    color_banner = {'None': '#2ecc71', 'Moderate': '#f39c12', 'Severe': '#e74c3c'}.get(s_name, 'gray')
    axes[row, 4].text(0.5, 0.6, f'Severity:\n{s_name}', ha='center', va='center', fontsize=14, fontweight='bold', color=color_banner)
    axes[row, 4].text(0.5, 0.25, f'Class:\n{c_name}', ha='center', va='center', fontsize=11)
    axes[row, 4].set_xlim(0, 1)
    axes[row, 4].set_ylim(0, 1)
    axes[row, 4].axis('off')

plt.suptitle('Chart 22: Preprocessed Multi-Modal Patch Gallery across Severity Levels', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Cross-Modal Relationships & Preprocessed Tensor Diagnostics
### Charts 23 through 26: Multi-Modal Bivariate Distributions & Tensor Verification
- **Chart 23**: Bivariate Hexbin / 2D Density plot: Spectral PC1 vs LiDAR Elevation.
- **Chart 24**: Violin Plots of LiDAR Elevation by Damage Severity level.
- **Chart 25**: Preprocessed Tensor Value Distribution ($X_{HSI}$ vs $X_{DEM}$).
- **Chart 26**: Comprehensive Dataset & Preprocessing Executive Metric Summary Card.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 23: Bivariate Spectral PC1 vs Elevation
flat_pc1 = hsi_pca[:, :, 0].flatten()[::50]
flat_dem = dem_raw.flatten()[::50]
valid_pair = np.isfinite(flat_dem)

hb = axes[0, 0].hexbin(flat_pc1[valid_pair], flat_dem[valid_pair], gridsize=35, cmap='inferno', mincnt=1)
axes[0, 0].set_title('Chart 23: Bivariate Density — Spectral PC1 vs LiDAR Elevation', fontweight='bold')
axes[0, 0].set_xlabel('Spectral Principal Component 1')
axes[0, 0].set_ylabel('Elevation (meters)')
plt.colorbar(hb, ax=axes[0, 0], label='Pixel Density')

# Chart 24: Elevation by Severity Violin Plot
patch_elev_means = np.mean(patches['X_dem'][:, 0], axis=(1, 2))
df_sev_elev = pd.DataFrame({
    'Severity': [SEVERITY_NAMES[s].capitalize() for s in patches['y_sev']],
    'Normalized Elevation': patch_elev_means
})
sns.violinplot(data=df_sev_elev, x='Severity', y='Normalized Elevation', palette={'None': '#2ecc71', 'Moderate': '#f39c12', 'Severe': '#e74c3c'}, ax=axes[0, 1], inner='box')
axes[0, 1].set_title('Chart 24: Patch Elevation Distribution across Damage Severity Levels', fontweight='bold')
axes[0, 1].set_ylabel('Mean Normalized Elevation [0, 1]')

# Chart 25: Preprocessed Tensor Distribution
axes[1, 0].hist(patches['X_hsi'].flatten()[::100], bins=50, color='royalblue', alpha=0.7, label='X_hsi (PCA Tensors)', density=True)
axes[1, 0].hist(patches['X_dem'].flatten()[::100], bins=50, color='crimson', alpha=0.6, label='X_dem (DEM Tensors)', density=True)
axes[1, 0].set_title('Chart 25: Preprocessed Tensor Value Distributions', fontweight='bold')
axes[1, 0].set_xlabel('Tensor Value')
axes[1, 0].set_ylabel('Probability Density')
axes[1, 0].legend()

# Chart 26: Executive Dataset Summary Card
axes[1, 1].axis('off')
summary_text = f'''==================================================\n           RUBICON PREPROCESSING & EDA SUMMARY\n==================================================\n* Dataset Source:         IEEE GRSS Houston 2013 (UTM 15N)\n* Spatial Dimensions:     {hsi_raw.shape[0]} Rows x {hsi_raw.shape[1]} Cols (~2.5m GSD)\n* Raw HSI Channels:       {hsi_raw.shape[2]} Bands (380 nm - 1050 nm)\n* PCA Reduced Channels:   {hsi_pca.shape[2]} Components\n* Retained Spectral Var:  {meta["pca_explained_variance"]*100:.2f}%\n* Preprocessed Patches:   {len(patches["y_class"])} Multi-Modal Tiles\n* Patch Spatial Extent:   {meta["patch_size"]} x {meta["patch_size"]} Pixels (~80m x 80m ground area)\n* LiDAR Elevation Range:  [{meta["dem_min_elevation"]:.2f} m, {meta["dem_max_elevation"]:.2f} m]\n* Land-Cover Classes:     15 Unique Categories\n* Damage Severity Proxy:  None: {meta["severity_distribution"].get("none", 0)} | Mod: {meta["severity_distribution"].get("moderate", 0)} | Sev: {meta["severity_distribution"].get("severe", 0)}\n* Preprocessed Artifacts: data/preprocessed/preprocessed_houston2013.npz\n=================================================='''

axes[1, 1].text(0.05, 0.5, summary_text, family='monospace', fontsize=10.5, va='center', bbox=dict(boxstyle='round,pad=0.8', facecolor='#f8f9fa', edgecolor='#bdc3c7', linewidth=1.5))
axes[1, 1].set_title('Chart 26: Executive Multi-Modal Dataset Summary Card', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n All EDA & Preprocessing Visualization Charts successfully rendered!')